# 中芯国际（SMIC）股票数据分析 Notebook## 688981.SH（A股科创板）+ 00981.HK（港股）近一年 K 线数据分析本 Notebook 完整演示了以下流程：1. **数据获取** — 分别获取 A 股和港股近一年的日线交易数据2. **数据清洗** — 解析、对齐两地交易日3. **可视化分析** — K 线图、成交量、归一化走势对比、AH 溢价率> 数据来源：通过 Tushare / 腾讯自选股接口获取  > 数据区间：2025-07-02 ~ 2026-07-02

In [ ]:
# ============================================# Step 1: 导入依赖库# ============================================# 如果缺少以下库，请先运行：# !pip install akshare plotly pandas numpyimport pandas as pdimport numpy as npimport jsonimport plotly.graph_objects as gofrom plotly.subplots import make_subplotsfrom datetime import datetime# 设置显示选项pd.set_option('display.max_rows', 20)pd.set_option('display.width', 200)print("✅ 所有库导入成功！")

In [ ]:
# ============================================# Step 2: 获取 A 股数据 (688981.SH)# ============================================# 方法一：使用 akshare（推荐，免费无需 token）# 若未安装，请运行: !pip install aksharetry:    import akshare as ak    # 获取中芯国际 A 股日线数据    a_raw = ak.stock_zh_a_hist(        symbol="688981",        period="daily",        start_date="20250702",        end_date="20260702",        adjust="qfq"  # 前复权    )    # 重命名列以匹配标准格式    a_df = a_raw.rename(columns={        '日期': 'date',        '开盘': 'open',        '收盘': 'close',        '最高': 'high',        '最低': 'low',        '成交量': 'volume',        '成交额': 'amount'    })    a_df['date'] = pd.to_datetime(a_df['date']).dt.strftime('%Y-%m-%d')    a_df = a_df[['date', 'open', 'close', 'high', 'low', 'volume', 'amount']]    a_df = a_df.sort_values('date').reset_index(drop=True)    print(f"📊 A 股数据获取成功！共 {len(a_df)} 个交易日")    print(f"   日期范围: {a_df['date'].iloc[0]} ~ {a_df['date'].iloc[-1]}")    print(f"   价格范围: ¥{a_df['low'].min():.2f} ~ ¥{a_df['high'].max():.2f}")except ImportError:    print("⚠️ akshare 未安装，使用本地 JSON 文件")    with open('中芯国际_sh688981_近一年数据.json', 'r', encoding='utf-8') as f:        a_data = json.load(f)    a_df = pd.DataFrame(a_data)    a_df['date'] = pd.to_datetime(a_df['date']).dt.strftime('%Y-%m-%d')    print(f"📊 从本地加载 A 股数据，共 {len(a_df)} 个交易日")a_df.head()

In [ ]:
# ============================================# Step 3: 获取港股数据 (00981.HK)# ============================================# 方法一：使用 aksharetry:    import akshare as ak    hk_raw = ak.stock_hk_hist(        symbol="00981",        period="daily",        start_date="20250702",        end_date="20260702",        adjust="qfq"    )    hk_df = hk_raw.rename(columns={        '日期': 'date',        '开盘': 'open',        '收盘': 'close',        '最高': 'high',        '最低': 'low',        '成交量': 'volume',        '成交额': 'amount'    })    hk_df['date'] = pd.to_datetime(hk_df['date']).dt.strftime('%Y-%m-%d')    hk_df = hk_df[['date', 'open', 'close', 'high', 'low', 'volume', 'amount']]    hk_df = hk_df.sort_values('date').reset_index(drop=True)    print(f"📊 港股数据获取成功！共 {len(hk_df)} 个交易日")    print(f"   日期范围: {hk_df['date'].iloc[0]} ~ {hk_df['date'].iloc[-1]}")    print(f"   价格范围: HK${hk_df['low'].min():.2f} ~ HK${hk_df['high'].max():.2f}")except ImportError:    print("⚠️ akshare 未安装，使用本地 JSON 文件")    with open('中芯国际_hk00981_近一年数据.json', 'r', encoding='utf-8') as f:        hk_data = json.load(f)    hk_df = pd.DataFrame(hk_data)    hk_df['date'] = pd.to_datetime(hk_df['date']).dt.strftime('%Y-%m-%d')    print(f"📊 从本地加载港股数据，共 {len(hk_df)} 个交易日")hk_df.head()

In [ ]:
# ============================================# Step 4: 基本统计分析# ============================================# A 股统计a_first = a_df['close'].iloc[0]a_last = a_df['close'].iloc[-1]a_chg = (a_last - a_first) / a_first * 100print("=" * 50)print("📈 中芯国际 A 股 (688981.SH) 近一年表现")print("=" * 50)print(f"  起始价: ¥{a_first:.2f}  →  最新价: ¥{a_last:.2f}")print(f"  涨跌幅: {a_chg:+.2f}%")print(f"  最高价: ¥{a_df['high'].max():.2f}（{a_df.loc[a_df['high'].idxmax(), 'date']}）")print(f"  最低价: ¥{a_df['low'].min():.2f}（{a_df.loc[a_df['low'].idxmin(), 'date']}）")print(f"  交易天数: {len(a_df)} 天")print(f"  日均成交量: {a_df['volume'].mean()/10000:.0f} 万手")# 计算 A 股波动率a_daily_ret = a_df['close'].pct_change().dropna()print(f"  日波动率(标准差): {a_daily_ret.std()*100:.2f}%")print()# 港股统计hk_first = hk_df['close'].iloc[0]hk_last = hk_df['close'].iloc[-1]hk_chg = (hk_last - hk_first) / hk_first * 100print("=" * 50)print("📈 中芯国际 港股 (00981.HK) 近一年表现")print("=" * 50)print(f"  起始价: HK${hk_first:.2f}  →  最新价: HK${hk_last:.2f}")print(f"  涨跌幅: {hk_chg:+.2f}%")print(f"  最高价: HK${hk_df['high'].max():.2f}（{hk_df.loc[hk_df['high'].idxmax(), 'date']}）")print(f"  最低价: HK${hk_df['low'].min():.2f}（{hk_df.loc[hk_df['low'].idxmin(), 'date']}）")print(f"  交易天数: {len(hk_df)} 天")print(f"  日均成交量: {hk_df['volume'].mean()/10000:.0f} 万股")hk_daily_ret = hk_df['close'].pct_change().dropna()print(f"  日波动率(标准差): {hk_daily_ret.std()*100:.2f}%")# 汇总对比表print()print("=" * 50)print("📊 两地表现对比")print("=" * 50)print(f"  {'指标':<16} {'A 股':>12} {'港股':>12}")print(f"  {'-'*40}")print(f"  {'起始价':<16} {'¥'+str(a_first):>12} {'HK$'+str(hk_first):>12}")print(f"  {'最新价':<16} {'¥'+str(a_last):>12} {'HK$'+str(hk_last):>12}")print(f"  {'涨跌幅':<16} {f'{a_chg:+.2f}%':>12} {f'{hk_chg:+.2f}%':>12}")print(f"  {'最高价':<16} {'¥'+str(a_df['high'].max()):>12} {'HK$'+str(hk_df['high'].max()):>12}")print(f"  {'最低价':<16} {'¥'+str(a_df['low'].min()):>12} {'HK$'+str(hk_df['low'].min()):>12}")print(f"  {'日波动率':<16} {f'{a_daily_ret.std()*100:.2f}%':>12} {f'{hk_daily_ret.std()*100:.2f}%':>12}")

In [ ]:
# ============================================# Step 5: A 股 K 线图 + 成交量# ============================================fig_a = make_subplots(    rows=2, cols=1,    shared_xaxes=True,    vertical_spacing=0.03,    row_heights=[0.7, 0.3],    subplot_titles=("A 股 K 线图 (688981.SH)", "成交量"))# K 线fig_a.add_trace(    go.Candlestick(        x=a_df['date'],        open=a_df['open'],        high=a_df['high'],        low=a_df['low'],        close=a_df['close'],        name='K线',        increasing={'line': {'color': '#e83939'}, 'fillcolor': '#e83939'},        decreasing={'line': {'color': '#2ba350'}, 'fillcolor': '#2ba350'},        showlegend=False    ),    row=1, col=1)# 成交量柱状图colors = ['#e83939' if c >= o else '#2ba350' for o, c in zip(a_df['open'], a_df['close'])]fig_a.add_trace(    go.Bar(        x=a_df['date'],        y=a_df['volume'],        name='成交量',        marker_color=colors,        showlegend=False    ),    row=2, col=1)fig_a.update_layout(    title='中芯国际 A 股 (688981.SH) — 近一年日线走势',    xaxis_title='日期',    yaxis_title='价格 (¥)',    yaxis2_title='成交量 (手)',    height=600,    template='plotly_white',    hovermode='x unified')fig_a.update_xaxes(rangeslider_visible=False)# 标记最高/最低价a_max_idx = a_df['high'].idxmax()a_min_idx = a_df['low'].idxmin()fig_a.add_annotation(    x=a_df['date'][a_max_idx], y=a_df['high'].max(),    text=f"最高 {a_df['high'].max():.1f}",    showarrow=True, arrowhead=1, font=dict(color='#e83939', size=10),    ay=-40)fig_a.add_annotation(    x=a_df['date'][a_min_idx], y=a_df['low'].min(),    text=f"最低 {a_df['low'].min():.1f}",    showarrow=True, arrowhead=1, font=dict(color='#2ba350', size=10),    ay=40)fig_a.show()

In [ ]:
# ============================================# Step 6: 港股 K 线图 + 成交量# ============================================fig_hk = make_subplots(    rows=2, cols=1,    shared_xaxes=True,    vertical_spacing=0.03,    row_heights=[0.7, 0.3],    subplot_titles=("港股 K 线图 (00981.HK)", "成交量"))fig_hk.add_trace(    go.Candlestick(        x=hk_df['date'],        open=hk_df['open'],        high=hk_df['high'],        low=hk_df['low'],        close=hk_df['close'],        name='K线',        increasing={'line': {'color': '#e83939'}, 'fillcolor': '#e83939'},        decreasing={'line': {'color': '#2ba350'}, 'fillcolor': '#2ba350'},        showlegend=False    ),    row=1, col=1)colors_hk = ['#e83939' if c >= o else '#2ba350' for o, c in zip(hk_df['open'], hk_df['close'])]fig_hk.add_trace(    go.Bar(        x=hk_df['date'],        y=hk_df['volume'],        name='成交量',        marker_color=colors_hk,        showlegend=False    ),    row=2, col=1)fig_hk.update_layout(    title='中芯国际 港股 (00981.HK) — 近一年日线走势',    xaxis_title='日期',    yaxis_title='价格 (HK$)',    yaxis2_title='成交量 (股)',    height=600,    template='plotly_white',    hovermode='x unified')fig_hk.update_xaxes(rangeslider_visible=False)fig_hk.show()

In [ ]:
# ============================================# Step 7: 对齐 A 股和港股数据# ============================================# 按共同交易日对齐a_dict = a_df.set_index('date')hk_dict = hk_df.set_index('date')common_dates = sorted(set(a_dict.index) & set(hk_dict.index))print(f"两地市场重叠交易日: {len(common_dates)} 天")print(f"A 股独立交易日: {len(a_df) - len(common_dates)} 天")print(f"港股独立交易日: {len(hk_df) - len(common_dates)} 天")# 构建对齐数据aligned = pd.DataFrame({    'date': common_dates,    'a_close': a_dict.loc[common_dates, 'close'].values,    'a_open': a_dict.loc[common_dates, 'open'].values,    'a_high': a_dict.loc[common_dates, 'high'].values,    'a_low': a_dict.loc[common_dates, 'low'].values,    'hk_close': hk_dict.loc[common_dates, 'close'].values,    'hk_open': hk_dict.loc[common_dates, 'open'].values,    'hk_high': hk_dict.loc[common_dates, 'high'].values,    'hk_low': hk_dict.loc[common_dates, 'low'].values,})# 归一化价格（以起始日=100）aligned['a_norm'] = aligned['a_close'] / aligned['a_close'].iloc[0] * 100aligned['hk_norm'] = aligned['hk_close'] / aligned['hk_close'].iloc[0] * 100# 计算 AH 溢价率（汇率按 0.91 估算）hkd_cny = 0.91aligned['premium'] = (aligned['a_close'] - aligned['hk_close'] * hkd_cny) / (aligned['hk_close'] * hkd_cny) * 100aligned.head(10)

In [ ]:
# ============================================# Step 8: 归一化价格走势对比# ============================================fig_norm = go.Figure()fig_norm.add_trace(go.Scatter(    x=aligned['date'], y=aligned['a_norm'],    mode='lines', name='A 股 (688981)',    line=dict(color='#e83939', width=2)))fig_norm.add_trace(go.Scatter(    x=aligned['date'], y=aligned['hk_norm'],    mode='lines', name='港股 (00981)',    line=dict(color='#2ba350', width=2)))# 添加 100 基准线fig_norm.add_hline(    y=100, line_dash="dash", line_color="gray",    annotation_text="起始基准线 = 100")fig_norm.update_layout(    title='归一化价格走势对比（以起始日收盘价为 100）',    xaxis_title='日期',    yaxis_title='指数 (起始=100)',    height=450,    template='plotly_white',    hovermode='x unified',    legend=dict(orientation='h', y=-0.15))fig_norm.show()# 关键结论a_final_norm = aligned['a_norm'].iloc[-1]hk_final_norm = aligned['hk_norm'].iloc[-1]print(f"📊 归一化指数:")print(f"   A 股: 100 → {a_final_norm:.1f}（{a_final_norm-100:+.1f}）")print(f"   港股: 100 → {hk_final_norm:.1f}（{hk_final_norm-100:+.1f}）")print(f"   👉 港股弹性更大，涨幅超出 A 股约 {hk_final_norm - a_final_norm:.1f} 个百分点")

In [ ]:
# ============================================# Step 9: AH 溢价率分析# ============================================avg_premium = aligned['premium'].mean()max_premium = aligned['premium'].max()min_premium = aligned['premium'].min()fig_premium = go.Figure()# 溢价率柱状图colors_prem = ['#e83939' if p >= 0 else '#2ba350' for p in aligned['premium']]fig_premium.add_trace(go.Bar(    x=aligned['date'], y=aligned['premium'],    name='溢价率',    marker_color=colors_prem,    showlegend=False))# 均值线fig_premium.add_hline(    y=avg_premium, line_dash="dash", line_color="black",    annotation_text=f"均值 {avg_premium:.1f}%")fig_premium.update_layout(    title=f'AH 溢价率走势（平均 {avg_premium:.1f}%）',    xaxis_title='日期',    yaxis_title='溢价率 (%)',    height=400,    template='plotly_white',    hovermode='x unified')fig_premium.show()print("=" * 50)print("📊 AH 溢价率统计")print("=" * 50)print(f"  平均溢价率: {avg_premium:.2f}%")print(f"  最高溢价率: {max_premium:.2f}%")print(f"  最低溢价率: {min_premium:.2f}%")print(f"  溢价率标准差: {aligned['premium'].std():.2f}%")print()print(f"  💡 A 股相对港股持续高溢价，均值约 90%，")print(f"     主要源于两地市场结构差异和流动性溢价。")

In [ ]:
# ============================================# Step 10: 双 K 线并排对比# ============================================sample_dates = aligned['date'].tolist()# 每 30 天取一个标签，避免 X 轴过密tick_vals = sample_dates[::30]tick_text = [d[5:] for d in tick_vals]  # MM-DD 格式fig_dual = make_subplots(    rows=2, cols=1,    shared_xaxes=True,    vertical_spacing=0.05,    row_heights=[0.5, 0.5],    subplot_titles=("A 股 K 线 (688981.SH)", "港股 K 线 (00981.HK)"))# A 股 K 线fig_dual.add_trace(    go.Candlestick(        x=aligned['date'], open=aligned['a_open'],        high=aligned['a_high'], low=aligned['a_low'],        close=aligned['a_close'], name='A股',        increasing={'line': {'color': '#e83939'}, 'fillcolor': '#e83939'},        decreasing={'line': {'color': '#2ba350'}, 'fillcolor': '#2ba350'},        showlegend=False    ),    row=1, col=1)# 港股 K 线fig_dual.add_trace(    go.Candlestick(        x=aligned['date'], open=aligned['hk_open'],        high=aligned['hk_high'], low=aligned['hk_low'],        close=aligned['hk_close'], name='港股',        increasing={'line': {'color': '#e83939'}, 'fillcolor': '#e83939'},        decreasing={'line': {'color': '#2ba350'}, 'fillcolor': '#2ba350'},        showlegend=False    ),    row=2, col=1)fig_dual.update_layout(    title='A 股 vs 港股 K 线并排对比（对齐交易日）',    height=700,    template='plotly_white',    hovermode='x unified')fig_dual.update_xaxes(rangeslider_visible=False)fig_dual.update_yaxes(title_text="价格 (¥)", row=1, col=1)fig_dual.update_yaxes(title_text="价格 (HK$)", row=2, col=1)fig_dual.show()

In [ ]:
# ============================================# Step 11: 日收益率分布对比# ============================================a_returns = aligned['a_close'].pct_change().dropna() * 100hk_returns = aligned['hk_close'].pct_change().dropna() * 100fig_ret = make_subplots(    rows=1, cols=2,    subplot_titles=("A 股日收益率分布", "港股日收益率分布"))fig_ret.add_trace(    go.Histogram(x=a_returns, nbinsx=40, name='A股',                 marker_color='#e83939', opacity=0.7, showlegend=False),    row=1, col=1)fig_ret.add_trace(    go.Histogram(x=hk_returns, nbinsx=40, name='港股',                 marker_color='#2ba350', opacity=0.7, showlegend=False),    row=1, col=2)fig_ret.update_layout(    title='日收益率分布对比',    height=350,    template='plotly_white',    showlegend=False)fig_ret.update_xaxes(title_text="日收益率 (%)", row=1, col=1)fig_ret.update_xaxes(title_text="日收益率 (%)", row=1, col=2)fig_ret.show()print("📊 日收益率统计:")print(f"  A 股: 均值 {a_returns.mean():.3f}%, 标准差 {a_returns.std():.3f}%")print(f"  港股: 均值 {hk_returns.mean():.3f}%, 标准差 {hk_returns.std():.3f}%")print(f"  👉 港股波动性更大（标准差 {hk_returns.std():.2f}% vs A 股 {a_returns.std():.2f}%）")

In [ ]:
# ============================================# Step 12: 总结# ============================================summary_data = {    '指标': ['起始价', '最新价', '涨跌幅', '最高价', '最低价', '交易天数', '日波动率', 'AH 平均溢价率', 'AH 最高溢价率', 'AH 最低溢价率'],    'A 股 (688981)': [        f'¥{a_first:.2f}', f'¥{a_last:.2f}', f'{a_chg:+.2f}%',        f'¥{a_df["high"].max():.2f}', f'¥{a_df["low"].min():.2f}',        f'{len(a_df)}天', f'{a_daily_ret.std()*100:.2f}%', '-', '-', '-'    ],    '港股 (00981)': [        f'HK${hk_first:.2f}', f'HK${hk_last:.2f}', f'{hk_chg:+.2f}%',        f'HK${hk_df["high"].max():.2f}', f'HK${hk_df["low"].min():.2f}',        f'{len(hk_df)}天', f'{hk_daily_ret.std()*100:.2f}%', '-', '-', '-'    ],}summary_df = pd.DataFrame(summary_data)# 补充 AH 溢价数据print("=" * 60)print("📊 中芯国际 AH 股综合分析总结")print("=" * 60)print()print(summary_df.to_string(index=False))print()print("AH 溢价分析:")print(f"  平均溢价率: {avg_premium:.2f}%")print(f"  最高溢价率: {max_premium:.2f}%  |  最低溢价率: {min_premium:.2f}%")print()print("📌 关键发现:")print(f"  1. 近一年两地均大幅上涨，A股 +{a_chg:.1f}%，港股 +{hk_chg:.1f}%")print(f"  2. 港股弹性更大，涨幅领先 A 股约 {hk_chg - a_chg:.1f} 个百分点")print(f"  3. A 股相对港股持续高溢价，平均 {avg_premium:.1f}%，最低也有 {min_premium:.1f}%")print(f"  4. 港股波动性显著高于 A 股（日标准差 {hk_daily_ret.std()*100:.2f}% vs {a_daily_ret.std()*100:.2f}%）")print(f"  5. 两地走势高度相关，重叠交易日达 {len(aligned)} 天")print()print("⚠️ 免责声明：以上分析仅基于历史数据，不构成任何投资建议。")print("   投资有风险，入市需谨慎。")